# Regression Algorithm for XGBOOST

In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor,plot_tree
import math

Creating a dataframe

In [3]:
CGPA=[6.7,9.0,7.5,5.0]
Package=[4.5,11.0,6.0,8.0]
df=pd.DataFrame({'CGPA':CGPA,'Package':Package})

In [4]:
df

,CGPA,Package
0,6.7,4.5
1,9.0,11.0
2,7.5,6.0
3,5.0,8.0


Model1 evaluate as average of all Package values

In [5]:
df['Model1']=np.average(df['Package'])
df['Res1']=df['Package']-df['Model1']

In [6]:
df = df.sort_values(by='CGPA').reset_index(drop=True)
df

,CGPA,Package,Model1,Res1
0,5.0,8.0,7.375,0.625
1,6.7,4.5,7.375,-2.875
2,7.5,6.0,7.375,-1.375
3,9.0,11.0,7.375,3.625


In [7]:
Reg_lambda = 0

In [8]:
# Function to calculate Similarity Score
def Similarity_Score(Residuals,Reg_lambda):
    return((np.sum(Residuals)**2)/(len(Residuals)+Reg_lambda))

In [9]:
Root_ss=Similarity_Score(df['Res1'],Reg_lambda)

In [10]:
split_points=[(df['CGPA'][i]+df['CGPA'][i+1])/2 for i in range(len(df['CGPA'])-1)]
split_points

[np.float64(5.85), np.float64(7.1), np.float64(8.25)]

In [11]:
best_gain = -1
best_split = None

In [12]:
for split in split_points:
    mask_left=df['CGPA']<=split
    mask_right=df['CGPA']>split

    left_res=df.loc[mask_left,'Res1'].values
    right_res=df.loc[mask_right,'Res1'].values

    ss_left=Similarity_Score(left_res,Reg_lambda)
    ss_right=Similarity_Score(right_res,Reg_lambda)

    gain=ss_left+ss_right-Root_ss

    if gain >best_gain:
        best_gain=gain
        best_split=split
print(f"Best Split Found at CGPA <= {best_split} with a Gain of {best_gain:.4f}\n")



Best Split Found at CGPA <= 8.25 with a Gain of 17.5208



In [13]:
mask_left=df['CGPA']<=best_split
mask_right=df['CGPA']>best_split

left_res=df.loc[mask_left,'Res1'].values
right_res=df.loc[mask_right,'Res1'].values


In [14]:
left_res


array([ 0.625, -2.875, -1.375])

In [15]:
right_res

array([3.625])

In [16]:
def leaf_output(residuals, reg_lambda):
    return np.sum(residuals) / (len(residuals) + reg_lambda)

left_output_val = leaf_output(left_res, Reg_lambda)
right_output_val = leaf_output(right_res, Reg_lambda)

In [17]:
left_output_val

np.float64(-1.2083333333333333)

In [18]:
right_output_val

np.float64(3.625)

Splitting left leaf

In [19]:
new_df = df[df['Res1'].isin(left_res)]
new_df

,CGPA,Package,Model1,Res1
0,5.0,8.0,7.375,0.625
1,6.7,4.5,7.375,-2.875
2,7.5,6.0,7.375,-1.375


In [20]:
split_points=[(new_df['CGPA'][i]+new_df['CGPA'][i+1])/2 for i in range(len(new_df['CGPA'])-1)]
split_points

[np.float64(5.85), np.float64(7.1)]

In [21]:
left_node_gain=gain
best_gain = -1
best_split = None

In [22]:
for split in split_points:
    mask_left=new_df['CGPA']<=split
    mask_right=new_df['CGPA']>split

    left_res=new_df.loc[mask_left,'Res1'].values
    right_res=new_df.loc[mask_right,'Res1'].values

    ss_left=Similarity_Score(left_res,Reg_lambda)
    ss_right=Similarity_Score(right_res,Reg_lambda)

    gain=ss_left+ss_right-Root_ss

    if gain >best_gain:
        best_gain=gain
        best_split=split
print(f"Best Split Found at CGPA <= {best_split} with a Gain of {best_gain:.4f}\n")



Best Split Found at CGPA <= 5.85 with a Gain of 9.4219



Model2=Model1+lr*Model2
where learning rate (lr) is 0.3

In [23]:
mask_left=new_df['CGPA']<=best_split
mask_right=new_df['CGPA']>best_split

left_res=new_df.loc[mask_left,'Res1'].values
right_res=new_df.loc[mask_right,'Res1'].values


In [24]:
left_res


array([0.625])

In [25]:
right_res

array([-2.875, -1.375])

In [26]:
def leaf_output(residuals, reg_lambda):
    return np.sum(residuals) / (len(residuals) + reg_lambda)

left_output_val = leaf_output(left_res, Reg_lambda)
right_output_val = leaf_output(right_res, Reg_lambda)

In [27]:
left_output_val

np.float64(0.625)

In [28]:
right_output_val

np.float64(-2.125)

# Classification Algorithm for XGBOOST

In [33]:
CGPA=[5.70,6.25,7.10,8.15,9.60]
Placed=[0,1,0,1,1]
df=pd.DataFrame({'CGPA':CGPA,'Placed':Placed})
df

,CGPA,Placed
0,5.70,0
1,6.25,1
2,7.10,0
3,8.15,1
4,9.60,1


In [34]:
zeros=(df['Placed']==0).sum()
ones=(df['Placed']==1).sum()
df['Pred_log_odds']=log_odds=math.log(ones/zeros)
df['Pred1_Prob']=1/(1+np.exp(-df['Pred_log_odds']))
df['Res1']=df['Placed']-df['Pred1_Prob']
df


,CGPA,Placed,Pred_log_odds,Pred1_Prob,Res1
0,5.70,0,0.405465,0.6,-0.6
1,6.25,1,0.405465,0.6,0.4
2,7.10,0,0.405465,0.6,-0.6
3,8.15,1,0.405465,0.6,0.4
4,9.60,1,0.405465,0.6,0.4


In [35]:
Reg_lambda = 0

In [36]:
# Function to calculate Similarity Score
def Similarity_Score(Residuals,Prevoius_Prob,Reg_lambda):
    return((np.sum(Residuals)**2)/(np.sum(Prevoius_Prob*(1-Prevoius_Prob))+Reg_lambda))

In [37]:
Root_ss=Similarity_Score(df['Res1'],df['Pred1_Prob'],Reg_lambda)

In [38]:
split_points=[(df['CGPA'][i]+df['CGPA'][i+1])/2 for i in range(len(df['CGPA'])-1)]
split_points

[np.float64(5.975), np.float64(6.675), np.float64(7.625), np.float64(8.875)]

In [39]:
best_gain = -1
best_split = None

In [41]:
for split in split_points:
    mask_left=df['CGPA']<=split
    mask_right=df['CGPA']>split

    left_res=df.loc[mask_left,'Res1'].values
    right_res=df.loc[mask_right,'Res1'].values

    ss_left=Similarity_Score(left_res,df['Pred1_Prob'],Reg_lambda)
    ss_right=Similarity_Score(right_res,df['Pred1_Prob'],Reg_lambda)

    gain=ss_left+ss_right-Root_ss

    if gain >best_gain:
        best_gain=gain
        best_split=split
print(f"Best Split Found at CGPA <= {best_split} with a Gain of {best_gain:.4f}\n")



Best Split Found at CGPA <= 7.625 with a Gain of 1.0667



In [42]:
mask_left=df['CGPA']<=best_split
mask_right=df['CGPA']>best_split

left_res=df.loc[mask_left,'Res1'].values
right_res=df.loc[mask_right,'Res1'].values


In [43]:
left_res


array([-0.6,  0.4, -0.6])

In [44]:
right_res

array([0.4, 0.4])

In [57]:
def leaf_output(Residuals,Prevoius_Prob,Reg_lambda):
    return((np.sum(Residuals))/(np.sum(Prevoius_Prob*(1-Prevoius_Prob))+Reg_lambda))

left_output_val = leaf_output(left_res,df['Pred1_Prob'], Reg_lambda)
right_output_val = leaf_output(right_res,df['Pred1_Prob'], Reg_lambda)

In [58]:
left_output_val

np.float64(-0.666666666666667)

In [59]:
right_output_val

np.float64(0.6666666666666665)

Log_odds2=m1+0.3*m2

In [64]:
0.405465+0.3*(left_output_val)

np.float64(0.20546499999999993)

In [65]:
0.405465+0.3*(right_output_val)


np.float64(0.6054649999999999)

Calculating Probablities

In [66]:
Log_odds_tranformed_left=1/(1+math.exp(-(0.405465+0.3*(left_output_val))))
Log_odds_tranformed_left


0.5511863037257506

In [67]:
Log_odds_tranformed_right=1/(1+math.exp(-(0.405465+0.3*(right_output_val))))

Log_odds_tranformed_right


0.646905614526508